# Persistent Memory

*Notebook 18*

Lesson 17 made conversation history durable.

This lesson makes it useful: keep stable facts, compress noise, and remove stale or sensitive data.


---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, SQLiteSession

import json
import asyncio

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 🎯 The Problem

A persistent session is still a transcript.

Useful memory is curated.

Keep durable facts and preferences.

Drop filler, stale details, and sensitive data.

---

## 📋 Part 1: Persisting a User Preference

Some stated preferences are useful durable context.

A file-backed session preserves the preference-setting turn.

A recreated session can load that turn for a later response.

In [ ]:
prefs_session = SQLiteSession(
    "user_prefs_demo",
    db_path="preferences.sqlite"
)

prefs_instructions = (
    "You are a helpful writing assistant.\n"
    "Remember the user's communication preferences and apply them in every response.\n"
    "If the user hasn't stated preferences yet, ask them before answering."
)

prefs_agent = Agent(
    name="PreferencesAssistant",
    instructions=prefs_instructions,
    model=MODEL
)

# Turn 1: user sets their preferences

result = await Runner.run(prefs_agent, input="I prefer short answers, bullet points over prose, and plain language (no jargon).", session=prefs_session)

print("Turn 1:", result.final_output)

#### Apply Preferences in a Later Session

In [ ]:
# Turn 2: recreate the session and inspect the stored preference
later_session = SQLiteSession(
    "user_prefs_demo",
    db_path="preferences.sqlite"
)

# Prove the preference reconnected from storage, before the model answers
stored = " ".join(
    str(item.get("content", ""))
    for item in await later_session.get_items()
    if item.get("role") == "user"
)
preference_loaded = all(t in stored for t in ("short answers", "bullet points", "plain language"))
print(f"Preference reconnected: {'yes' if preference_loaded else 'no'}")

result = await Runner.run(prefs_agent, input="Explain what a REST API is.", session=later_session)

print("Turn 2:", result.final_output)

### 💡 Key Takeaway

The recreated session contains the saved preference turn.

The model's response shows how it used that context in this run.

---

## ⚠️ Part 2: Controlling Session Growth

Sessions store more items after every turn.

Later runs resend more history, increasing input tokens.

Without a limit or compaction strategy, the history can exceed the context window.

Compact before that happens.

#### Build a Noisy Session

In [ ]:
# General-purpose assistant used by the demos below
assistant = Agent(
    name="Assistant",
    instructions="You are a helpful assistant. Remember useful facts users share about themselves.",
    model=MODEL
)

# Build a noisy session with many turns
noisy_session = SQLiteSession(
    "user_noisy_demo",
    db_path="noisy.sqlite"
)

messages = [
    "I'm building a Python web scraper.",
    "I want to scrape product prices from e-commerce sites.",
    "I prefer using requests and BeautifulSoup.",
    "I've already handled pagination.",
    "The main challenge left is rate limiting.",
]

for message in messages:

    await Runner.run(assistant, input=message, session=noisy_session)

items_before = await noisy_session.get_items()
print(f"Before cleanup: {len(items_before)} items")

#### Summarize the History

In [ ]:
# Step 1: Summarize, extract only what's worth keeping
summarizer = Agent(
    name="Summarizer",
    instructions="Extract only the key facts worth remembering. Return a concise bullet list.",
    model=MODEL
)

history_text = json.dumps(items_before, indent=2)

summary_result = await Runner.run(summarizer, input=f"Summarize the durable facts in this conversation:\n{history_text}")

summary = summary_result.final_output
print(f"Summary:\n{summary}")

#### Clear and Store the Summary

In [ ]:
# Step 2: Validate the summary retained the key facts BEFORE clearing the source history.
# Match on robust concept substrings, not exact phrases: the summarizer paraphrases.
required_terms = ("scrap", "limit")  # scraping, rate-limiting
summary_is_usable = all(term in summary.lower() for term in required_terms)
print(f"Summary kept key facts: {'yes' if summary_is_usable else 'no'}")

if summary_is_usable:
    # Store ONLY the summary with add_items()
    await noisy_session.clear_session()

    await noisy_session.add_items([
        {"role": "user", "content": f"Conversation summary:\n{summary}"}
    ])

    items_after = await noisy_session.get_items()
    print(f"After compaction: {len(items_after)} item (was {len(items_before)})")

    # Ask the model to use the compacted memory: recall a fact from the summary
    result = await Runner.run(assistant, input="What's the main challenge left in my project?", session=noisy_session)

    print(f"\nRecall test: {result.final_output}")
else:
    print("⚠️ Summary is missing a key fact. Keeping the original history instead of compacting.")

### 💡 Key Takeaway

Compaction trades detail for a smaller context: review or test the summary before replacing the source history.

Apply the pattern proactively, on a turn-count threshold, rather than waiting for quality to degrade.

For OpenAI Responses API models, the SDK also provides `OpenAIResponsesCompactionSession` for automated compaction.

## 🔐 Part 3: What NOT to Store: Privacy and Correction

Persistent memory is powerful but requires discipline.

Not everything belongs in long-term storage, and stored facts can become wrong over time.

#### What to Keep vs Drop

<div style="text-align: left; display: inline-block;">

| Keep | Drop |
|------|------|
| Name, role, goals | Greetings and filler turns |
| Stated preferences | Repeated confirmations |
| Decisions and constraints | Stale intermediate outputs |
| Durable facts | Low-value tool noise |

</div>

⚠️ **Security note:** Never pass passwords, API keys, financial details, or health information into a persistent session.

Redact sensitive data in application code before calling `Runner.run()`.

In [ ]:
correction_session = SQLiteSession(
    "user_correction_demo",
    db_path="correction.sqlite"
)

correction_instructions = (
    "You are a helpful assistant. Remember what users tell you about themselves. "
    "If a user corrects an earlier fact, treat the most recent statement as the current truth."
)

correction_agent = Agent(
    name="CorrectionAssistant",
    instructions=correction_instructions,
    model=MODEL
)

# Store an initial fact

result = await Runner.run(correction_agent, input="I work as a data analyst.", session=correction_session)

print(f"Stored: {result.final_output}\n")

# User corrects the fact

result = await Runner.run(correction_agent, input="Actually I just changed jobs, I'm now a machine learning engineer.", session=correction_session)

print(f"Corrected: {result.final_output}\n")

# Ask which statement the agent treats as current

result = await Runner.run(correction_agent, input="What's my current role?", session=correction_session)

print(f"Answer in this run: {result.final_output}")

# Prove both statements remain in storage
stored_user = " ".join(
    str(item.get("content", ""))
    for item in await correction_session.get_items()
    if item.get("role") == "user"
).lower()
print(f"\nBoth facts still in SQLite: data analyst={'data analyst' in stored_user}, "
      f"machine learning engineer={'machine learning engineer' in stored_user}")

#### Conflicting Memory and Forgetting

The instructions ask the agent to treat the latest statement as current.

Both statements remain in SQLite: `SQLiteSession` stores items, not semantic facts.

To remove a fact buried in history, rebuild the session without it: summarize, drop the unwanted fact, clear, and store.

For repeated conflicts, apply that pattern rather than letting contradictions stack.

### 💡 Key Takeaway

Treat memory as a curated record.

Keep stable facts and preferences.

Drop noise and sensitive data.

Correct stale facts before contradictions accumulate.

---

## 🧹 Demo Cleanup

In [ ]:
# Close open connections before deleting (open SQLite files can lock on some systems)
for db_session in (prefs_session, later_session, noisy_session, correction_session):
    await asyncio.to_thread(db_session.close)

# Delete the SQLite databases AND their WAL sidecar files.
# These files ARE the persistent storage. Delete them for a completely fresh start.
for db in ["preferences.sqlite", "noisy.sqlite", "correction.sqlite"]:
    for suffix in ("", "-wal", "-shm"):
        Path(f"{db}{suffix}").unlink(missing_ok=True)
    print(f"✅ Removed {db} (+ sidecars)")
print("🧹 Demo cleanup complete")

---

## 💪 Practice Exercises

### Exercise 1: Curated User Profile

*Covers: curation, compacting a transcript into durable facts*

Collect user facts across several turns, then compact them into a profile that survives a session rebuild.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Curated User Profile
# --------------------------------------------------------------
# Objective: Compact a conversation into a durable user profile.
#
# TODO 1: Create a file-backed SQLiteSession: store the db_path in a variable
# TODO 2: Collect a name, a preference, and a goal across several turns
# TODO 3: Read the stored items with get_items()
# TODO 4: Summarize only the durable profile facts
# TODO 5: Before clearing, validate the summary contains the name, preference,
#          and goal. Only then clear the transcript and add_items() the summary
# TODO 6: Recreate the session and inspect get_items() to confirm those exact
#          facts survived
# TODO 7: Close both session objects (asyncio.to_thread(session.close)), then
#          delete the db_path file plus its -wal and -shm sidecars

# --- Write your code below this line ---

### Exercise 2: Memory Cleanup

*Covers: memory management: summarize, clear, and store*

Build a session with 5+ turns, then apply the summarize → clear → store pattern.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Memory Cleanup
# --------------------------------------------------------------
# Objective: Apply the summarize → clear → store pattern.
#
# TODO 1: Create a file-backed SQLiteSession with a stored db_path, then run
#          5 turns of conversation on any topic
# TODO 2: Call get_items() and print how many items are stored
# TODO 3: Run a summarizer agent to extract key facts
# TODO 4: Validate the summary contains one required fact, then clear_session()
# TODO 5: Use add_items() to store only the summary, then confirm the count dropped
# TODO 6: Inspect get_items() to confirm the summary was stored, then ask the
#          agent to use it in one response
# TODO 7: Close the session (asyncio.to_thread(session.close)), then delete
#          the session database and its -wal and -shm sidecars

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Persistence and curation are different jobs**

- A file-backed session gives you a durable transcript (Lesson 17)

- Curation turns that transcript into useful memory: facts, preferences, decisions

- Inspect `get_items()` to prove what was stored, not just that the model recalled it
<br>
<br>

**Compaction trades detail for room**

- summarize → clear → `add_items()` replaces history with a compact summary

- A model-written summary can drop or distort details: review it before relying on it
<br>
<br>

**Sensitive and stale data need application policy**

- Redact secrets before `Runner.run()`: nothing in the session layer filters them for you

- Instructions can tell the model to prefer the newest fact

- Both versions remain in SQLite until you rebuild the session
<br>
<br>

**Sessions retrieve by ID, vector memory retrieves by meaning**

- `SQLiteSession` returns one conversation's items by session ID

- Lesson 19 stores individual memories as embeddings and retrieves the closest match

---

## 📍 Next Step

**Notebook 19: Vector Memory with ChromaDB**  

Store individual memories as embeddings, then retrieve the ones closest in meaning to a query.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-18-persistent-memory)

---